## Training: CLIP multimodal with offline augmentation (v2.1)

This notebook extends the CLIP baseline by generating **offline image augmentations** for minority classes, then training a CLIP-based classifier.

### Inputs
- `CLEANDFPATH`: clean CSV from the download/validation pipeline
- `IMAGEDIR`: cached base images folder

### What “offline augmentation” means here
- For each minority class, we randomly transform some existing images and **save them to disk** in `AUG_DIR`.
- The augmented rows are appended to the training dataframe (validation/test stay unchanged).

### Outputs (saved to `SAVE_DIR`)
- Augmented training CSV (`TRAIN_AUG_CSV_PATH`)
- Best/final model weights + config JSON + test predictions


In [ ]:
!pip -q install -U transformers accelerate scikit-learn torchvision

In [ ]:
import os
import json
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from torchvision import transforms

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torch.optim.lr_scheduler as lr_scheduler

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from transformers import AutoProcessor, CLIPModel
from tqdm.auto import tqdm

In [ ]:
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 1) Setup + load clean data + split

We load the cleaned dataframe, filter to rows with images, and create stratified train/val/test splits.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

def first_existing_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return candidates[0]

CLEANDF_CANDIDATES = [
    "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/clean_df.csv",
]

IMAGEDIR_CANDIDATES = [
    "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/images",
]

CLEANDFPATH = first_existing_path(CLEANDF_CANDIDATES)
IMAGEDIR = first_existing_path(IMAGEDIR_CANDIDATES)

SAVE_DIR = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/clip_multimodal_offline_aug"
AUG_DIR = os.path.join(SAVE_DIR, "offline_aug_images")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(AUG_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(SAVE_DIR, "clip_multimodal_best.pth")
FINAL_MODEL_PATH = os.path.join(SAVE_DIR, "clip_multimodal_final.pth")
CONFIG_PATH = os.path.join(SAVE_DIR, "clip_multimodal_config.json")
TEST_PRED_PATH = os.path.join(SAVE_DIR, "clip_multimodal_test_predictions.csv")
TRAIN_AUG_CSV_PATH = os.path.join(SAVE_DIR, "train_with_offline_aug.csv")
SPLIT_INFO_PATH = os.path.join(SAVE_DIR, "split_sizes.json")

print("CLEANDFPATH:", CLEANDFPATH, os.path.exists(CLEANDFPATH))
print("IMAGEDIR:", IMAGEDIR, os.path.exists(IMAGEDIR))
print("SAVE_DIR:", SAVE_DIR)
print("AUG_DIR:", AUG_DIR)

In [ ]:
df = pd.read_csv(CLEANDFPATH)

def pick_first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None

TEXTCOL = pick_first_existing(df.columns, ["clean_title", "clean title", "cleantitle", "title"])
LABELCOL = pick_first_existing(df.columns, ["6_way_label", "6 way label", "6waylabel", "label"])
IDCOL = pick_first_existing(df.columns, ["id"])

assert TEXTCOL is not None, "No text column found."
assert LABELCOL is not None, "No label column found."
assert IDCOL is not None, "No id column found."

def find_image_path(image_id, image_dir):
    for ext in [".jpg", ".jpeg", ".png", ".webp"]:
        candidate = os.path.join(image_dir, f"{image_id}{ext}")
        if os.path.exists(candidate):
            return candidate
    return None

df = df.copy()
df[TEXTCOL] = df[TEXTCOL].fillna("").astype(str)
df[LABELCOL] = df[LABELCOL].astype(int)
df[IDCOL] = df[IDCOL].astype(str)

df["image_path"] = df[IDCOL].apply(lambda x: find_image_path(x, IMAGEDIR))
df = df[df["image_path"].notnull()].reset_index(drop=True)

print("Filtered shape:", df.shape)
print(df[LABELCOL].value_counts().sort_index())

In [ ]:
RANDOM_STATE = 42

df_train, df_test = train_test_split(
    df,
    test_size=0.20,
    stratify=df[LABELCOL],
    random_state=RANDOM_STATE
)

df_test, df_val = train_test_split(
    df_test,
    test_size=0.50,
    stratify=df_test[LABELCOL],
    random_state=RANDOM_STATE
)

df_train = df_train.reset_index(drop=True).copy()
df_val = df_val.reset_index(drop=True).copy()
df_test = df_test.reset_index(drop=True).copy()

print("Train:", df_train.shape)
print("Val:", df_val.shape)
print("Test:", df_test.shape)

with open(SPLIT_INFO_PATH, "w") as f:
    json.dump({
        "train_size_before_aug": int(len(df_train)),
        "val_size": int(len(df_val)),
        "test_size": int(len(df_test)),
        "text_col": TEXTCOL,
        "label_col": LABELCOL,
        "id_col": IDCOL,
    }, f, indent=2)

## 2) Offline augmentation for minority classes

Goal: reduce class imbalance by *increasing the number of training images* for minority labels.

We pick a target count per class (`TARGET_PER_CLASS`). For each class below this target, we:
- sample existing rows with replacement
- apply random image transforms
- save augmented images to `AUG_DIR`
- append the new rows into `df_train_final`

Only the **training** set is augmented (val/test remain unchanged).


In [ ]:
train_counts = df_train[LABELCOL].value_counts().sort_index()
NUM_CLASSES = len(sorted(df[LABELCOL].unique()))

TARGET_PER_CLASS = 1200
minority_classes = train_counts[train_counts < TARGET_PER_CLASS].index.tolist()

print("Original train counts:")
print(train_counts)
print("Minority classes:", minority_classes)
print("Target per class:", TARGET_PER_CLASS)

In [ ]:
offline_aug = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.90, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.10),
    transforms.RandomRotation(5),
])

In [ ]:
def save_augmented_pil(pil_img, save_path):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    pil_img.save(save_path, format="JPEG", quality=95)

augmented_rows = []

for cls in minority_classes:
    cls_df = df_train[df_train[LABELCOL] == cls].reset_index(drop=True)
    current_count = len(cls_df)
    needed = max(0, TARGET_PER_CLASS - current_count)

    print(f"Class {cls}: current={current_count}, need_to_add={needed}")

    if needed == 0:
        continue

    for k in tqdm(range(needed), desc=f"Offline augment class {cls}"):
        row = cls_df.sample(n=1, replace=True, random_state=SEED + k).iloc[0]

        orig_image_path = row["image_path"]
        orig_id = str(row[IDCOL])

        img = Image.open(orig_image_path).convert("RGB")
        aug_img = offline_aug(img)

        new_sample_id = f"{orig_id}__augcls{cls}_{k}"
        save_path = os.path.join(AUG_DIR, f"class_{cls}", f"{new_sample_id}.jpg")
        save_augmented_pil(aug_img, save_path)

        new_row = row.copy()
        new_row["image_path"] = save_path
        new_row["sample_id"] = new_sample_id
        new_row["is_augmented"] = 1
        new_row["source_image_id"] = orig_id

        augmented_rows.append(new_row)

df_train = df_train.copy()
df_train["sample_id"] = df_train[IDCOL].astype(str)
df_train["is_augmented"] = 0
df_train["source_image_id"] = df_train[IDCOL].astype(str)

if len(augmented_rows) > 0:
    df_aug = pd.DataFrame(augmented_rows)
    df_train_final = pd.concat([df_train, df_aug], ignore_index=True)
else:
    df_aug = pd.DataFrame(columns=df_train.columns)
    df_train_final = df_train.copy()

df_train_final = df_train_final.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print("\nTrain counts after offline augmentation:")
print(df_train_final[LABELCOL].value_counts().sort_index())

df_train_final.to_csv(TRAIN_AUG_CSV_PATH, index=False)
print("Saved offline-augmented training CSV to:", TRAIN_AUG_CSV_PATH)

In [ ]:
CLIP_NAME = "openai/clip-vit-base-patch32"
MAX_LEN = 77
BATCH_SIZE = 16

processor = AutoProcessor.from_pretrained(CLIP_NAME)

print("CLIP_NAME:", CLIP_NAME)
print("MAX_LEN:", MAX_LEN)
print("BATCH_SIZE:", BATCH_SIZE)
print("NUM_CLASSES:", NUM_CLASSES)

In [ ]:
# train_counts = df_train[LABELCOL].value_counts().sort_index()
# minority_classes = train_counts[train_counts < 1200].index.tolist()

# print("Train counts:")
# print(train_counts)
# print("Minority classes:", minority_classes)

# train_img_tfms = transforms.Compose([
#     transforms.RandomResizedCrop(224, scale=(0.90, 1.0)),
#     transforms.RandomHorizontalFlip(p=0.5),
#     transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.10),
#     transforms.RandomRotation(5),
# ])

In [ ]:
class FakedditCLIPDataset(Dataset):
    def __init__(self, df, textcol, labelcol, sample_id_col="sample_id"):
        self.df = df.reset_index(drop=True)
        self.textcol = textcol
        self.labelcol = labelcol
        self.sample_id_col = sample_id_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        text = str(row[self.textcol])
        label = int(row[self.labelcol])
        sample_id = str(row[self.sample_id_col])
        image_path = row["image_path"]

        image = Image.open(image_path).convert("RGB")

        return {
            "text": text,
            "label": label,
            "sample_id": sample_id,
            "image": image
        }

def clip_collate_fn(batch):
    texts = [item["text"] for item in batch]
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    sample_ids = [item["sample_id"] for item in batch]
    images = [item["image"] for item in batch]

    enc = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LEN
    )

    enc["labels"] = labels
    enc["sample_id"] = sample_ids
    enc["text"] = texts
    return enc

In [ ]:
if "sample_id" not in df_val.columns:
    df_val = df_val.copy()
    df_val["sample_id"] = df_val[IDCOL].astype(str)

if "sample_id" not in df_test.columns:
    df_test = df_test.copy()
    df_test["sample_id"] = df_test[IDCOL].astype(str)

train_dataset = FakedditCLIPDataset(df_train_final, TEXTCOL, LABELCOL, sample_id_col="sample_id")
val_dataset = FakedditCLIPDataset(df_val, TEXTCOL, LABELCOL, sample_id_col="sample_id")
test_dataset = FakedditCLIPDataset(df_test, TEXTCOL, LABELCOL, sample_id_col="sample_id")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    collate_fn=clip_collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=clip_collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=clip_collate_fn
)

sample = next(iter(train_loader))
print("input_ids", sample["input_ids"].shape)
print("attention_mask", sample["attention_mask"].shape)
print("pixel_values", sample["pixel_values"].shape)
print("labels", sample["labels"].shape)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

train_labels = df_train_final[LABELCOL].to_numpy()

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
print("Class weights:", class_weights)

In [ ]:
class CLIPMultimodalClassifier(nn.Module):
    def __init__(self, clip_name="openai/clip-vit-base-patch32", num_classes=6, dropout=0.3):
        super().__init__()

        self.clip = CLIPModel.from_pretrained(clip_name)
        proj_dim = self.clip.config.projection_dim

        self.norm = nn.LayerNorm(proj_dim * 4)
        self.classifier = nn.Sequential(
            nn.Linear(proj_dim * 4, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

        self.freeze_backbone()

    def forward(self, input_ids, attention_mask, pixel_values):
        outputs = self.clip(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values
        )

        image_feat = F.normalize(outputs.image_embeds, p=2, dim=-1)
        text_feat = F.normalize(outputs.text_embeds, p=2, dim=-1)

        abs_diff = torch.abs(image_feat - text_feat)
        elem_mul = image_feat * text_feat

        fused = torch.cat([image_feat, text_feat, abs_diff, elem_mul], dim=-1)
        fused = self.norm(fused)
        logits = self.classifier(fused)
        return logits

    def freeze_backbone(self):
        for p in self.clip.parameters():
            p.requires_grad = False
        for p in self.norm.parameters():
            p.requires_grad = True
        for p in self.classifier.parameters():
            p.requires_grad = True

    def unfreeze_top_layers(self, vision_last_n=2, text_last_n=2):
        self.freeze_backbone()

        for layer in self.clip.vision_model.encoder.layers[-vision_last_n:]:
            for p in layer.parameters():
                p.requires_grad = True

        for layer in self.clip.text_model.encoder.layers[-text_last_n:]:
            for p in layer.parameters():
                p.requires_grad = True

    def unfreeze_all(self):
        for p in self.parameters():
            p.requires_grad = True

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
class EarlyStoppingMax:
    def __init__(self, patience=3, verbose=True, delta=1e-4, savepath=None):
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.savepath = savepath
        self.counter = 0
        self.best_score = None
        self.earlystop = False

    def __call__(self, score, model):
        if self.best_score is None or score > self.best_score + self.delta:
            self.best_score = score
            self.counter = 0
            if self.savepath is not None:
                torch.save(model.state_dict(), self.savepath)
        else:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.earlystop = True

def make_optimizer(model, head_lr=1e-3, backbone_lr=2e-5, weight_decay=1e-4):
    head_params, backbone_params = [], []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("clip."):
            backbone_params.append(p)
        else:
            head_params.append(p)

    param_groups = []
    if head_params:
        param_groups.append({"params": head_params, "lr": head_lr})
    if backbone_params:
        param_groups.append({"params": backbone_params, "lr": backbone_lr})

    return torch.optim.AdamW(param_groups, weight_decay=weight_decay)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    pbar = tqdm(loader, total=len(loader), desc="Training", leave=True)

    for batch in pbar:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        pixel_values = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values
        )

        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / len(loader.dataset)

def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            pixel_values = batch["pixel_values"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values
            )

            loss = criterion(logits, labels)
            preds = torch.argmax(logits, dim=1)

            running_loss += loss.item() * labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss = running_loss / len(loader.dataset)
    val_acc = accuracy_score(all_labels, all_preds)
    val_macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    val_weighted_f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

    return val_loss, val_acc, val_macro_f1, val_weighted_f1

In [ ]:
model = CLIPMultimodalClassifier(
    clip_name=CLIP_NAME,
    num_classes=NUM_CLASSES,
    dropout=0.3
).to(device)

print("Trainable params after initial freeze:", count_trainable_params(model))

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

optimizer = make_optimizer(
    model,
    head_lr=1e-3,
    backbone_lr=2e-5,
    weight_decay=1e-4
)

scheduler = lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=1,
    min_lr=1e-6
)

num_epochs = 5
head_only_epochs = 2

earlystopping = EarlyStoppingMax(
    patience=2,
    verbose=True,
    delta=1e-4,
    savepath=BEST_MODEL_PATH
)

history = {
    "epoch": [],
    "stage": [],
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
    "val_macro_f1": [],
    "val_weighted_f1": [],
    "lr_head": [],
    "lr_backbone": [],
}

for epoch in range(num_epochs):
    if epoch == head_only_epochs:
        model.unfreeze_top_layers(vision_last_n=2, text_last_n=2)
        optimizer = make_optimizer(
            model,
            head_lr=5e-4,
            backbone_lr=2e-5,
            weight_decay=1e-4
        )
        scheduler = lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=1,
            min_lr=1e-6
        )
        print("Unfroze top 2 text and vision encoder layers.")
        print("Trainable params now:", count_trainable_params(model))

    stage_name = "head_only" if epoch < head_only_epochs else "top_layers"

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_macro_f1, val_weighted_f1 = validate_one_epoch(
        model, val_loader, criterion, device
    )

    scheduler.step(val_macro_f1)
    earlystopping(val_macro_f1, model)

    lrs = [pg["lr"] for pg in optimizer.param_groups]
    lr_head = lrs[0] if len(lrs) > 0 else None
    lr_backbone = lrs[1] if len(lrs) > 1 else None

    history["epoch"].append(epoch + 1)
    history["stage"].append(stage_name)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_macro_f1"].append(val_macro_f1)
    history["val_weighted_f1"].append(val_weighted_f1)
    history["lr_head"].append(lr_head)
    history["lr_backbone"].append(lr_backbone)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Stage {stage_name} | "
        f"Train Loss {train_loss:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"Val Acc {val_acc:.4f} | "
        f"Val Macro F1 {val_macro_f1:.4f} | "
        f"Val Weighted F1 {val_weighted_f1:.4f} | "
        f"LRs {lrs}"
    )

    if earlystopping.earlystop:
        print("Early stopping triggered.")
        break

torch.save(model.state_dict(), FINAL_MODEL_PATH)
print("Final model saved to:", FINAL_MODEL_PATH)
print("Best model saved to:", BEST_MODEL_PATH)

In [ ]:
hist_df = pd.DataFrame(history)

plt.figure(figsize=(16, 4))

plt.subplot(1, 4, 1)
plt.plot(hist_df["epoch"], hist_df["train_loss"], label="train_loss")
plt.plot(hist_df["epoch"], hist_df["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss")
plt.legend()

plt.subplot(1, 4, 2)
plt.plot(hist_df["epoch"], hist_df["val_acc"], label="val_acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy")
plt.legend()

plt.subplot(1, 4, 3)
plt.plot(hist_df["epoch"], hist_df["val_macro_f1"], label="val_macro_f1")
plt.plot(hist_df["epoch"], hist_df["val_weighted_f1"], label="val_weighted_f1")
plt.xlabel("Epoch")
plt.ylabel("F1")
plt.title("Validation F1")
plt.legend()

plt.subplot(1, 4, 4)
plt.plot(hist_df["epoch"], hist_df["lr_head"], label="lr_head")
if hist_df["lr_backbone"].notnull().any():
    plt.plot(hist_df["epoch"], hist_df["lr_backbone"], label="lr_backbone")
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.title("Learning Rates")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
class FakedditCLIPDataset(Dataset):
    def __init__(self, df, textcol, labelcol, idcol, image_transform=None, augment_classes=None):
        self.df = df.reset_index(drop=True)
        self.textcol = textcol
        self.labelcol = labelcol
        self.idcol = idcol
        self.image_transform = image_transform
        self.augment_classes = set(augment_classes or [])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        text = str(row[self.textcol])
        label = int(row[self.labelcol])
        imageid = str(row[self.idcol])
        imagepath = row["image_path"]

        image = Image.open(imagepath).convert("RGB")
        if self.image_transform is not None and label in self.augment_classes:
            image = self.image_transform(image)

        return {
            "text": text,
            "label": label,
            "imageid": imageid,
            "image": image,
        }

## 3) Train CLIP model (freeze → unfreeze top layers)

Training schedule:
- Start with CLIP backbone frozen (train normalization + classifier only)
- After `head_only_epochs`, unfreeze the top encoder layers (text+vision) and continue fine-tuning

Model selection uses **validation macro-F1** (helps minority classes).


In [ ]:
def clip_collate_fn(batch):
    texts = [item["text"] for item in batch]
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    imageids = [item["imageid"] for item in batch]
    images = [item["image"] for item in batch]

    proc = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
    )

    enc = {k: v for k, v in proc.items()}   # plain dict, not BatchFeature
    enc["labels"] = labels
    enc["imageid"] = imageids
    enc["text"] = texts
    return enc

In [ ]:
best_model = CLIPMultimodalClassifier(
    clip_name=CLIP_NAME,
    num_classes=NUM_CLASSES,
    dropout=0.3
).to(device)

best_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
best_model.eval()

print("Loaded best checkpoint.")

In [ ]:
# Evaluation reminder:
# - Reuse the same dataset/collate setup as training
# - Recreate loaders (especially if you changed batch size / num_workers)
# - Rebuild the model with the SAME architecture/hyperparams used during training
best_model = CLIPMultimodalClassifier(
    clip_name=CLIP_NAME,
    num_classes=NUM_CLASSES,
    dropout=0.2   # must match the trained version
).to(device)

best_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
best_model.eval()

# Run evaluation only (no training here)
test_loss, test_acc, test_precision, test_recall, test_weighted_f1, test_macro_f1, pred_df, all_labels, all_preds = evaluate_model(
    best_model, test_loader, criterion, device
)

In [ ]:
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test F1: {test_weighted_f1:.4f}")
print(f"Test F1: {test_macro_f1:.4f}")

In [ ]:
print(classification_report(all_labels, all_preds, digits=4))

In [ ]:
def evaluate_model(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []
    all_imageids, all_texts = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            pixel_values = batch["pixel_values"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values
            )

            loss = criterion(logits, labels)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            bs = labels.size(0)

            running_loss += loss.item() * bs
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

            batch_imageids = batch.get("imageid", [None] * bs)
            batch_texts = batch.get("text", [""] * bs)

            all_imageids.extend(batch_imageids)
            all_texts.extend(batch_texts)

    avg_loss = running_loss / len(loader.dataset)
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
    recall = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
    weighted_f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    results_df = pd.DataFrame({
        "imageid": all_imageids,
        "text": all_texts,
        "true_label": all_labels,
        "pred_label": all_preds
    })

    prob_array = np.array(all_probs)
    for i in range(prob_array.shape[1]):
        results_df[f"prob_class_{i}"] = prob_array[:, i]

    return avg_loss, accuracy, precision, recall, weighted_f1, macro_f1, results_df, all_labels, all_preds

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()

In [ ]:
pred_df.to_csv(TEST_PRED_PATH, index=False)

config = {
    "clip_name": CLIP_NAME,
    "max_len": MAX_LEN,
    "batch_size": BATCH_SIZE,
    "num_classes": NUM_CLASSES,
    "text_col": TEXTCOL,
    "label_col": LABELCOL,
    "id_col": IDCOL,
    "clean_df_path": CLEANDFPATH,
    "image_dir": IMAGEDIR,
    "best_model_path": BEST_MODEL_PATH,
    "final_model_path": FINAL_MODEL_PATH,
    "test_predictions_path": TEST_PRED_PATH,
    "save_dir": SAVE_DIR,
    "seed": SEED,
    "minority_classes": minority_classes,
}

with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)

print("Saved predictions to:", TEST_PRED_PATH)
print("Saved config to:", CONFIG_PATH)